# 377. Combination Sum IV

## Topic Alignment
- Counting permutations with repetition appears in sequence modeling, dynamic programming over ordered choices, and analyzing decision trees where order matters (e.g., A/B test sequences, multi-step user journeys).

## Metadata 摘要
- Source: https://leetcode.com/problems/combination-sum-iv/
- Tags: Dynamic Programming, Array, Complete Knapsack, Counting
- Difficulty: Medium
- Priority: High

## Problem Statement 原题描述
Given an array of **distinct** integers `nums` and a target integer `target`, return the **number of possible combinations** that add up to `target`.

The test cases are generated so that the answer can fit in a **32-bit** integer.

**Note**: Different sequences are counted as different combinations.

**Constraints**:
- 1 <= nums.length <= 200
- 1 <= nums[i] <= 1000
- All the elements of nums are unique
- 1 <= target <= 1000

## Progressive Hints
- Hint 1: Despite the name "combination", this problem actually counts **permutations** (order matters).
- Hint 2: Use dp[i] to represent the number of ways to make target i.
- Hint 3: For permutations, iterate target in outer loop, nums in inner loop.
- Hint 4: For each target value, try all possible last numbers: dp[target] += dp[target - num].
- Hint 5: Initialize dp[0] = 1 (one way to make 0: use no numbers).

## Solution Overview
**Important**: Despite the problem title, this counts **permutations**, not combinations!
- [1, 2] and [2, 1] are counted as different

**Key Difference from LC 518 (Coin Change II)**:
- **LC 518 (combinations)**: Outer loop = coins, Inner loop = amounts
- **LC 377 (permutations)**: Outer loop = amounts, Inner loop = nums

**State**: `dp[i]` = number of permutations that sum to i

**Recurrence**: 
```python
for each num in nums:
    dp[target] += dp[target - num]
```

## Detailed Explanation

### Permutations vs Combinations

**Example**: nums = [1, 2], target = 3

**Permutations (this problem)**:
1. [1, 1, 1]
2. [1, 2]
3. [2, 1]
- Total: **3 permutations**

**Combinations (LC 518)**:
1. [1, 1, 1]
2. [1, 2] (same as [2, 1])
- Total: **2 combinations**

---

### Loop Order: The Critical Difference

**Permutations (this problem) - Outer loop: targets**:
```python
for target in range(1, max_target + 1):  # Outer: iterate targets
    for num in nums:                      # Inner: iterate nums
        if target >= num:
            dp[target] += dp[target - num]
```
- **Why this works**: For each target, we try all numbers as the "next" number
- No fixed number order
- Different orderings counted separately
- Example: When computing dp[3], we consider both "2 then 1" and "1 then 2"

**Combinations (LC 518) - Outer loop: nums**:
```python
for num in nums:                      # Outer: iterate nums
    for target in range(num, max_target + 1):  # Inner: iterate targets
        dp[target] += dp[target - num]
```
- **Why this works**: Process numbers in fixed order
- When processing num N, only use combinations from nums {1, 2, ..., N-1}
- Ensures each combination counted once

---

### Why Does Loop Order Create This Difference?

**Permutations (target outer)**:
- For target T, we ask: "What if the last number is N?"
- We try all possible N from nums
- No constraint on previous numbers
- Result: [A, B] and [B, A] both counted

**Combinations (nums outer)**:
- For num N, we ask: "How many ways to build targets using nums up to N?"
- When adding N, we only use combinations built from {nums[0], ..., nums[N-1]}
- This enforces a canonical order
- Result: Only one of [A, B] or [B, A] is counted

---

### Example Walkthrough

**Input**: nums = [1, 2, 3], target = 4

**Initial**: `dp = [1, 0, 0, 0, 0]` (dp[0]=1)

**target = 1**:
- Try num=1: dp[1] += dp[0] = 1
- Try num=2: skip (2 > 1)
- Try num=3: skip (3 > 1)
- `dp = [1, 1, 0, 0, 0]` (way: [1])

**target = 2**:
- Try num=1: dp[2] += dp[1] = 1
- Try num=2: dp[2] += dp[0] = 1+1 = 2
- Try num=3: skip
- `dp = [1, 1, 2, 0, 0]` (ways: [1,1], [2])

**target = 3**:
- Try num=1: dp[3] += dp[2] = 2
- Try num=2: dp[3] += dp[1] = 2+1 = 3
- Try num=3: dp[3] += dp[0] = 3+1 = 4
- `dp = [1, 1, 2, 4, 0]` (ways: [1,1,1], [1,2], [2,1], [3])

**target = 4**:
- Try num=1: dp[4] += dp[3] = 4
- Try num=2: dp[4] += dp[2] = 4+2 = 6
- Try num=3: dp[4] += dp[1] = 6+1 = 7
- `dp = [1, 1, 2, 4, 7]`

**Result**: dp[4] = 7

**The 7 permutations are**:
1. [1, 1, 1, 1]
2. [1, 1, 2]
3. [1, 2, 1]
4. [2, 1, 1]
5. [1, 3]
6. [3, 1]
7. [2, 2]

---

### Complete Knapsack: Combinations vs Permutations Summary

| Aspect | Combinations (LC 518) | Permutations (LC 377) |
|--------|----------------------|----------------------|
| Outer Loop | Items (coins/nums) | Capacity (amount/target) |
| Inner Loop | Capacity | Items |
| Order Matters? | No: [1,2] = [2,1] | Yes: [1,2] ≠ [2,1] |
| Direction | Left to right | Left to right |
| Use Case | Count sets | Count sequences |

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Backtracking | O(target^n) | O(target) | Exponential, too slow |
| Top-down DP + memo | O(target × n) | O(target) | Recursive with cache |
| Bottom-up DP | O(target × n) | O(target) | Most efficient |

In [ ]:
class Solution:
    def combinationSum4(self, nums: list[int], target: int) -> int:
        """
        Complete Knapsack - counting permutations.
        
        Time: O(target × n)
        Space: O(target)
        """
        # dp[i] = number of permutations to make target i
        dp = [0] * (target + 1)
        dp[0] = 1  # One way to make 0: use no numbers
        
        # IMPORTANT: Outer loop = target, Inner loop = nums
        # This gives PERMUTATIONS (order matters)
        for t in range(1, target + 1):
            for num in nums:
                if t >= num:
                    # Add number of ways to make (t - num)
                    dp[t] += dp[t - num]
        
        return dp[target]

In [ ]:
# Test cases
tests = [
    ([1, 2, 3], 4, 7),         # 7 permutations as shown above
    ([9], 3, 0),               # Impossible
    ([1], 1, 1),               # Single number
    ([1], 2, 1),               # [1,1]
    ([1, 2], 3, 3),            # [1,1,1], [1,2], [2,1]
    ([2, 1, 3], 35, 1132436852),  # Large case
    ([4, 2, 1], 32, 39882198), # Another large case
]

solver = Solution()
for nums, target, expected in tests:
    result = solver.combinationSum4(nums, target)
    assert result == expected, f"Failed for nums={nums}, target={target}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(target × n) where n = len(nums)
  - For each value from 1 to target, iterate through all nums
  - Total iterations: target × n
- **Space**: O(target)
  - Single DP array of size target + 1
  - Recursion with memoization would use O(target) for call stack + cache

## Edge Cases & Pitfalls
- **target = 0**: Return 1 (one way: use no numbers)
- **All nums > target**: Return 0 (impossible)
- **Single number**: If target % num == 0, return 1; else return 0
- **Large results**: Problem guarantees answer fits in 32-bit int
- **Loop order**: CRITICAL! Must be outer=target for permutations
- **Misleading name**: Problem title says "combination" but actually counts permutations
- **Comparison with LC 518**: Same DP structure, different loop order
- **Integer overflow**: With constraints (target ≤ 1000), result can be very large but guaranteed to fit

## Follow-up Variants
- **Follow-up in problem**: What if negative numbers are allowed? Need to detect negative cycles.
- **Count combinations**: See LC 518 (swap loop order)
- **Limited uses**: Each number can be used at most k times
- **Ordered subset**: Find kth permutation in lexicographic order
- **With constraints**: Additional constraints like "must use at least one of each number"
- **Print all permutations**: Reconstruct actual permutations, not just count

## Takeaways
- **Misleading Title**: Problem says "combination" but counts **permutations** (order matters)
- **Loop Order is Everything**: Outer=target gives permutations, outer=items gives combinations
- **Permutations Pattern**: Process targets in outer loop to count all orderings
- **Complete Knapsack**: Items can be reused unlimited times (left to right traversal)
- **Compare LC 518 vs LC 377**: Same DP template, only loop order differs
- **State Definition**: dp[i] = number of ways (permutations) to reach target i
- **Base Case**: dp[0] = 1 is fundamental to all knapsack counting problems

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 518 | Coin Change II | Complete knapsack combinations |
| LC 322 | Coin Change | Complete knapsack minimization |
| LC 279 | Perfect Squares | Complete knapsack minimization |
| LC 39 | Combination Sum | Backtracking variant |
| LC 70 | Climbing Stairs | Simple DP counting |